# Save-As-You-Go: Streaming DAQ Data to HDF5

This notebook demonstrates versionable's incremental HDF5 persistence.
Data is written to disk as it arrives — no need to hold everything in memory.

> **Requirements:** `plotly` and `jupyterlab` — install with `pixi install`

In [ ]:
from __future__ import annotations

from collections.abc import Iterator
from dataclasses import dataclass, field
from datetime import UTC, datetime
from pathlib import Path
from typing import Annotated

import numpy as np
import numpy.typing as npt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import versionable
import versionable.hdf5
from versionable import Appendable, Versionable

## 1. Define the Recording type

A normal Versionable dataclass. The `Appendable` annotation marks `time_s` and
`voltage_V` as growable — inside a session, they become resizable HDF5 datasets.

In [ ]:
@dataclass
class Recording(Versionable, version=1, hash="74a182"):
    """A single DAQ recording with time-series data."""

    name: str = ""
    sampleRate_Hz: float = 0.0
    startTime: datetime = field(default_factory=lambda: datetime.now(tz=UTC))
    time_s: Annotated[npt.NDArray[np.float64], Appendable()] = field(default_factory=lambda: np.empty(0))
    voltage_V: Annotated[npt.NDArray[np.float64], Appendable()] = field(default_factory=lambda: np.empty(0))

## 2. DAQ Simulator

A simple generator that yields chunks of a sinusoidal signal, simulating
a real data acquisition device.

In [ ]:
class DaqSimulator:
    """Generates a sinusoidal signal in chunks."""

    def __init__(
        self,
        frequency_Hz: float = 10.0,
        amplitude_V: float = 1.0,
        sampleRate_Hz: float = 1000.0,
        chunkSize: int = 100,
        duration_s: float = 1.0,
    ) -> None:
        self.frequency_Hz = frequency_Hz
        self.amplitude_V = amplitude_V
        self.sampleRate_Hz = sampleRate_Hz
        self.chunkSize = chunkSize
        self.duration_s = duration_s

    def stream(self, startOffset_s: float = 0.0) -> Iterator[tuple[np.ndarray, np.ndarray]]:
        """Yield (time_chunk, voltage_chunk) tuples."""
        dt = 1.0 / self.sampleRate_Hz
        totalSamples = int(self.duration_s * self.sampleRate_Hz)
        sampleIdx = 0

        while sampleIdx < totalSamples:
            n = min(self.chunkSize, totalSamples - sampleIdx)
            t = startOffset_s + (sampleIdx + np.arange(n)) * dt
            v = self.amplitude_V * np.sin(2 * np.pi * self.frequency_Hz * t)
            yield t, v
            sampleIdx += n

## 3. Record the first session

`open()` accepts either a **class** (empty proxy) or an **existing instance**.
Here we pass an instance — only set what you care about, defaults handle the rest.

In [ ]:
OUTPUT_PATH = Path("recording.h5")

daq = DaqSimulator(frequency_Hz=10.0, amplitude_V=1.0, sampleRate_Hz=1000.0, duration_s=1.0)

# Initialize the data (Optional) — Appendable fields and startTime have sensible defaults
rec = Recording(name="10 Hz sinusoid", sampleRate_Hz=daq.sampleRate_Hz)

with versionable.hdf5.open(rec, OUTPUT_PATH, mode="overwrite") as rec:
    for tChunk, vChunk in daq.stream():
        rec.time_s.append(tChunk)
        rec.voltage_V.append(vChunk)

print(f"Wrote {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size / 1024:.1f} KB)")

## 4. Load and plot the first session

The file is a standard HDF5 — load it with `versionable.load()`, no special API.

In [ ]:
first = versionable.load(Recording, OUTPUT_PATH)

print(f"Name:        {first.name}")
print(f"Sample rate: {first.sampleRate_Hz:.0f} Hz")
print(f"Start time:  {first.startTime.isoformat()}")
print(f"Samples:     {len(first.time_s)}")
print(f"Duration:    {first.time_s[-1] - first.time_s[0]:.3f} s")

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=first.time_s,
        y=first.voltage_V,
        mode="lines",
        name="First session",
        line=dict(color="#636EFA"),
    )
)
fig.update_layout(
    title="After first session (1.0 s)",
    xaxis_title="Time (s)",
    yaxis_title="Voltage (V)",
    template="plotly_white",
    height=350,
)
fig.show()

## 5. Resume and append more data

Reopen with `mode="resume"` — existing data is restored, and new appends
continue from where we left off.

In [ ]:
# Figure out where the first session ended
prev = versionable.load(Recording, OUTPUT_PATH)
startOffset = float(prev.time_s[-1]) + 1.0 / prev.sampleRate_Hz
print(f"Resuming from {len(prev.time_s)} samples (t={startOffset:.4f} s)")

# Append another 0.5 seconds
daq2 = DaqSimulator(frequency_Hz=10.0, amplitude_V=1.0, sampleRate_Hz=1000.0, duration_s=0.5)

with versionable.hdf5.open(Recording, OUTPUT_PATH, mode="resume") as rec:
    for tChunk, vChunk in daq2.stream(startOffset_s=startOffset):
        rec.time_s.append(tChunk)
        rec.voltage_V.append(vChunk)

    print(f"Total samples: {len(rec.time_s)}")

## 6. Load and plot the combined result

The file now contains both sessions. The boundary is seamless.

In [ ]:
combined = versionable.load(Recording, OUTPUT_PATH)
boundary = len(first.time_s)

print(f"Total samples: {len(combined.time_s)}")
print(f"Total duration: {combined.time_s[-1] - combined.time_s[0]:.3f} s")

fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("After first session (1.0 s)", "After resume (+0.5 s)"),
    vertical_spacing=0.15,
)

# Top: first session only
fig.add_trace(
    go.Scatter(
        x=first.time_s,
        y=first.voltage_V,
        mode="lines",
        name="First session",
        line=dict(color="#636EFA"),
    ),
    row=1,
    col=1,
)

# Bottom: both sessions, colored differently
fig.add_trace(
    go.Scatter(
        x=combined.time_s[:boundary],
        y=combined.voltage_V[:boundary],
        mode="lines",
        name="First session",
        line=dict(color="#636EFA"),
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=combined.time_s[boundary:],
        y=combined.voltage_V[boundary:],
        mode="lines",
        name="Resumed session",
        line=dict(color="#EF553B"),
    ),
    row=2,
    col=1,
)

fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_yaxes(title_text="V", row=1, col=1)
fig.update_yaxes(title_text="V", row=2, col=1)
fig.update_layout(template="plotly_white", height=550)
fig.show()

## Cleanup

In [ ]:
OUTPUT_PATH.unlink()  # Delete the file when done
print(f"Deleted {OUTPUT_PATH.name}")